# Notebook 31 - Consistently normalised comparison (N1 to N4)

**Why this runs before the second corpus.** The external review asked the one question the record could not answer: how much of the reported recovery-seed effect is unstable BatchNorm state, and how much survives when every model in the comparison receives the same deployment-time recalibration. The factorial students were never saved, so this notebook reruns the five-by-five factorial on both original architectures with every final checkpoint kept, applies one fixed recalibration procedure to all of them, and recomputes every quantity of the factorial analysis under both normalisations: variance shares, permutation tests, rank stability, REI, paired contrasts, and the ground-truth endpoints.

**Three controls ride along.** The dense, unpruned model recovered under the identical subsets, orders, weighting and schedule (is pruning necessary for the collapse); recalibration alone on the frozen raw students of the 17b and 20b registries (does any no-gradient procedure make raw pruning deployable); and a dual-mode evaluation, on validation only, of the frozen shallow random 40% student that escalates 11.8% of benign test traffic. The NB30 remedy is re-run on its saved snapshots with the full outcome set (attack-miss rate, per-family recall, ground-truth risk).

**Pre-registered claims** (stage 2, fixed before any result): N1 seed effect survives recalibration in at least two of four cells; N2 no method effect becomes detectable that was not detectable as-trained; N3 the unpruned model never collapses; N4 recalibration alone rescues no raw student. A claim that does not hold is reported as such.

**Checkpoints** are written to `results/saber/31_consistent_normalisation/checkpoints/` with SHA-256 in `checkpoint_registry.csv`; add that folder to `.gitignore` before committing, the registry is what goes in the repo.

Stages 4 and 6 are resumable per cell. Frozen structures from the 17b/20b registries. No new selection. No test access. GPU required; stage 4's shallow full-recovery cells are the heavy part.

In [ ]:
# Stage 1 - bootstrap
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, sys, json, copy, hashlib, itertools
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, yaml
import matplotlib.pyplot as plt

REPO = Path("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression")
assert REPO.exists(), f"repo not found: {REPO}"
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.saber.bridge_ciciot import load_bridge
from src.saber.taxonomy import ciciot2023_taxonomy, DEFAULT_COST_PROFILES
from src.saber.surgery import prune_cnn1d_channels
from src.saber.metrics import full_model_audit, action_weighted_boundary_inversion_rate

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
R = REPO / "results/saber"
OUT = R / "31_consistent_normalisation"
OUT.mkdir(parents=True, exist_ok=True)
CKPT_DIR = OUT / "checkpoints"; CKPT_DIR.mkdir(exist_ok=True)   # weights stay out of git; SHA-256 goes in the registry
print("repo:", REPO, "| device:", DEVICE)


In [ ]:
# Stage 2 - pre-registration
#
# The external review asked the one question the record could not answer: how much of the
# reported recovery-seed effect is unstable BatchNorm state, and how much survives when every
# model in the comparison receives the same deployment-time recalibration. The factorial
# students were never saved, so this reruns the factorial with every final checkpoint kept,
# applies ONE fixed recalibration procedure to all of them, and recomputes everything under
# both normalisations. Three cheap controls ride along. Criteria are fixed here, before any
# result exists, and a claim that does not hold is reported as such.

PREREG = {
    "arm": "N_consistent_normalisation",
    "recalibration": ("reset BatchNorm running statistics; forward 50 batches of 1,024 rows from ONE "
                      "fixed training-data calibration slice (seed 2026, identical for every model) in "
                      "train mode with cumulative averaging and no gradient; evaluate in eval mode"),
    "factorial": ("5 methods x 5 seeds, frozen 40% structures from the 17b/20b registries; shallow "
                  "regimes minimal_ce (1 unit on a seeded 10% subset) and full_ce (6 fixed full "
                  "epochs); deep regimes minimal_ce and standard_ce (2 full epochs); Adam 1e-3, "
                  "batch 1024, inverse-sqrt class weighting; final checkpoint of every cell saved"),
    "claims": {
        "N1": ("after uniform recalibration, a recovery-seed effect on AWBIR remains detectable by "
               "permutation (p < 0.05) in at least two of the four architecture-regime cells"),
        "N2": ("after uniform recalibration, no selection-method effect on AWBIR becomes detectable "
               "(p < 0.05) in a cell where it was not detectable as-trained"),
        "N3": ("the dense, unpruned model recovered under the identical subsets, orders, weighting and "
               "schedule produces no normalisation collapse (eval-mode benign escalation > 0.10 with "
               "batch-statistics escalation <= 0.10) in any of five seeds on either architecture"),
        "N4": ("recalibration alone, with no gradient step, brings no frozen raw student below 10% "
               "benign escalation; if it does for some, Section 7.2's claim is narrowed to "
               "'without recalibration'")},
    "also_recorded": [
        "dual-mode evaluation on the validation subsample of the frozen shallow random 40% student "
        "that escalates 11.8% of benign test traffic (the test partition is not touched)",
        "NB30 remedy re-run on its saved snapshots with the full outcome set: attack-miss rate, "
        "per-family recall, ground-truth HSR under all three profiles, ECE, before and after"],
    "seeds": [101, 211, 307, 401, 503], "no_test_access": True, "no_new_selection": True,
}
(OUT / "N_PREREGISTRATION.json").write_text(json.dumps(PREREG, indent=2))
print(json.dumps(PREREG, indent=2))

METHODS = ["random", "magnitude", "taylor", "fisher", "saber_v2"]
SEEDS = PREREG["seeds"]
REGIMES = {"shallow": {"minimal_ce": ("subset", 1), "full_ce": ("full", 6)},
           "deep":    {"minimal_ce": ("subset", 1), "standard_ce": ("full", 2)}}
RUN_CELLS = [("shallow", "minimal_ce"), ("deep", "minimal_ce"), ("deep", "standard_ce"), ("shallow", "full_ce")]
SUBSET_FRACTION = 0.10
COLLAPSE_B2A = 0.10
CAL_SEED = 2026


In [ ]:
# Stage 3 - data, teachers, structures, helpers
TRAIN_LOADER, VAL_LOADER, _TEST_UNUSED, SHALLOW_TEACHER, CLASS_NAMES = load_bridge()
taxonomy = ciciot2023_taxonomy(CLASS_NAMES)
robust_graph = pd.read_csv(R / "14_risk_graph/asvg_edges_robust.csv")
N_CLASSES = len(CLASS_NAMES)
SABER_CFG = yaml.safe_load(open(REPO / "config/saber.yaml"))
MIN_W = {"shallow": int(SABER_CFG["groups"]["minimum_remaining_per_layer"]), "deep": 8}
FAM = np.asarray(taxonomy.class_to_family_index)
FAMILIES = list(taxonomy.families)


class DeepCNN1D(nn.Module):
    def __init__(self, n_classes=34):
        super().__init__()
        def blk(i, o):
            return [nn.Conv1d(i, o, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(o)]
        self.conv = nn.Sequential(*blk(1, 64), *blk(64, 128), nn.MaxPool1d(2),
                                  *blk(128, 128), *blk(128, 256))
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Linear(256, n_classes)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        return self.head(self.pool(self.conv(x.float())).squeeze(-1))


TEACHERS = {"shallow": SHALLOW_TEACHER.to(DEVICE).eval()}
_dt = DeepCNN1D(N_CLASSES)
_dt.load_state_dict(torch.load(REPO / "models/ciciot2023/deepcnn1d_g5_seed0.pt",
                               map_location="cpu", weights_only=False)["state_dict"])
TEACHERS["deep"] = _dt.to(DEVICE).eval()

Xv, Yv = VAL_LOADER.dataset.tensors
VAL_Y_ALL = Yv.numpy()
_rng = np.random.default_rng(0)
_idx = np.concatenate([_rng.permutation(np.where(VAL_Y_ALL == c)[0])[:4000]
                       for c in range(N_CLASSES) if (VAL_Y_ALL == c).sum() > 0])
_idx = np.random.default_rng(12345).permutation(_idx)
EX_X = Xv[_idx].to(DEVICE); EX_Y = VAL_Y_ALL[_idx]
EXAMPLE_INPUT = Xv[:8].float().to(DEVICE)

_train_y = TRAIN_LOADER.dataset.tensors[1].numpy()
_counts = np.bincount(_train_y, minlength=N_CLASSES)
_w = np.zeros_like(_counts, dtype=np.float64)
_w[_counts > 0] = 1.0 / np.sqrt(_counts[_counts > 0]); _w[_counts > 0] /= _w[_counts > 0].mean()
CLASS_W = torch.tensor(_w, dtype=torch.float32, device=DEVICE)
N_TRAIN = len(TRAIN_LOADER.dataset)

# ONE fixed calibration slice for every model in this notebook
_cal_idx = torch.randperm(N_TRAIN, generator=torch.Generator().manual_seed(CAL_SEED))[:50 * 1024]
CAL_LOADER = torch.utils.data.DataLoader(
    torch.utils.data.Subset(TRAIN_LOADER.dataset, _cal_idx.tolist()), batch_size=1024, shuffle=False)


def forward_logits(model):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(EX_X[i:i + 8192]).cpu() for i in range(0, len(EX_X), 8192)]).numpy()


def forward_logits_batch_stats(model):
    saved = {n: (m.running_mean.clone(), m.running_var.clone(), m.momentum, m.num_batches_tracked.clone())
             for n, m in model.named_modules() if isinstance(m, nn.BatchNorm1d) and m.running_mean is not None}
    model.train()
    for n, m in model.named_modules():
        if n in saved:
            m.momentum = 0.0
    with torch.no_grad():
        out = torch.cat([model(EX_X[i:i + 8192]).cpu() for i in range(0, len(EX_X), 8192)]).numpy()
    for n, m in model.named_modules():
        if n in saved:
            rm, rv, mom, nbt = saved[n]
            m.running_mean.copy_(rm); m.running_var.copy_(rv); m.momentum = mom; m.num_batches_tracked.copy_(nbt)
    model.eval()
    return out


T_LOGITS = {a: forward_logits(m) for a, m in TEACHERS.items()}
T_PRED = {a: T_LOGITS[a].argmax(1) for a in TEACHERS}


def audit_from_logits(lg, arch):
    a = full_model_audit(lg, EX_Y, taxonomy, DEFAULT_COST_PROFILES)
    aw, _ = action_weighted_boundary_inversion_rate(T_LOGITS[arch], lg, EX_Y, robust_graph)
    p = lg.argmax(1)
    s_fam_ok = FAM[p] == FAM[EX_Y]; t_fam_ok = FAM[T_PRED[arch]] == FAM[EX_Y]
    out = {"b2a": float(a["benign_to_attack_rate"]), "a2b": float(a["attack_to_benign_rate"]),
           "family_f1": float(a["family_macro_f1"]), "fine_f1": float(a["fine_macro_f1"]),
           "awbir": float(aw), "ece": float(a["ece15"]),
           "hsr_miss": float(a["hsr_miss_sensitive"]), "hsr_balanced": float(a["hsr_balanced_soc"]),
           "hsr_fatigue": float(a["hsr_alert_fatigue"]),
           "teacher_correction_family": float(np.mean(s_fam_ok & ~t_fam_ok)),
           "teacher_degradation_family": float(np.mean(~s_fam_ok & t_fam_ok))}
    for fi, fname in enumerate(FAMILIES):
        mask = FAM[EX_Y] == fi
        out[f"recall_{fname}"] = float(np.mean(FAM[p][mask] == fi)) if mask.any() else float("nan")
    return out


def audit(model, arch, mode="eval"):
    lg = forward_logits(model) if mode == "eval" else forward_logits_batch_stats(model)
    return audit_from_logits(lg, arch)


def set_bn_momentum(model, momentum):
    for m in model.modules():
        if isinstance(m, nn.BatchNorm1d):
            m.momentum = momentum


def recalibrate_bn(model, n_batches=50):
    for m in model.modules():
        if isinstance(m, nn.BatchNorm1d):
            m.reset_running_stats(); m.momentum = None
    model.train()
    with torch.no_grad():
        for i, (xb, _) in enumerate(CAL_LOADER):
            if i >= n_batches:
                break
            model(xb.float().to(DEVICE))
    set_bn_momentum(model, 0.1)
    model.eval()
    return model


def make_subset_loader(seed):
    gen = torch.Generator().manual_seed(seed)
    sub = torch.randperm(N_TRAIN, generator=gen)[: int(N_TRAIN * SUBSET_FRACTION)]
    return torch.utils.data.DataLoader(torch.utils.data.Subset(TRAIN_LOADER.dataset, sub.tolist()),
                                       batch_size=1024, shuffle=True, generator=torch.Generator().manual_seed(seed))


def removed_path(arch, method, budget=0.40):
    if arch == "shallow":
        return R / f"17b_calibrated_checkpoint_freeze/{method}_r{int(round(budget * 100))}cal_removed_groups.csv"
    return R / f"20b_depth_checkpoint_freeze/{method}_minimal_r40_removed_groups.csv"


def raw_student(arch, method, budget=0.40):
    rp = removed_path(arch, method, budget)
    assert rp.exists(), f"frozen structure not found for {arch}/{method}/{budget}: {rp}"
    rm = pd.read_csv(rp)
    pm = {str(l): sorted(g["channel_index"].astype(int).tolist()) for l, g in rm.groupby("module_path")}
    st, _ = prune_cnn1d_channels(TEACHERS[arch], pm, EXAMPLE_INPUT, minimum_remaining_per_layer=MIN_W[arch])
    return st.to(DEVICE)


# runtime proofs: the probe is non-destructive and discriminative; recalibration touches only BN buffers
for _a, _m in TEACHERS.items():
    _b = {n: (x.running_mean.clone(), x.running_var.clone()) for n, x in _m.named_modules() if isinstance(x, nn.BatchNorm1d)}
    _e1 = forward_logits(_m); _p = forward_logits_batch_stats(_m); _e2 = forward_logits(_m)
    for n, x in _m.named_modules():
        if isinstance(x, nn.BatchNorm1d):
            assert torch.equal(x.running_mean, _b[n][0]) and torch.equal(x.running_var, _b[n][1])
    assert np.allclose(_e1, _e2, atol=1e-5) and not _m.training
    assert not np.allclose(_e1, _p, atol=1e-6), f"{_a}: probe indistinguishable from eval mode"
_t = copy.deepcopy(TEACHERS["shallow"]); _w0 = {n: p.detach().clone() for n, p in _t.named_parameters()}
recalibrate_bn(_t)
for n, p in _t.named_parameters():
    assert torch.equal(p, _w0[n]), f"recalibration altered weight {n}"
print("probe and recalibration verified on:", list(TEACHERS), "| evaluation rows:", len(_idx),
      "| calibration slice rows:", len(_cal_idx))


In [ ]:
# Stage 4 - factorial with checkpoints, evaluated as-trained and after uniform recalibration
RUNS = OUT / "factorial_runs.csv"
REG = OUT / "checkpoint_registry.csv"
run_rows = pd.read_csv(RUNS).to_dict("records") if RUNS.exists() else []
reg_rows = pd.read_csv(REG).to_dict("records") if REG.exists() else []
done = {(r["architecture"], r["regime"], r["method"], r["seed"]) for r in run_rows}
print("complete cells:", len(done))


def recover_cell(arch, method, regime, seed):
    kind, n_ep = REGIMES[arch][regime]
    torch.manual_seed(seed); np.random.seed(seed)
    student = raw_student(arch, method)
    raw = audit(student, arch, "eval")
    loader = make_subset_loader(seed) if kind == "subset" else torch.utils.data.DataLoader(
        TRAIN_LOADER.dataset, batch_size=1024, shuffle=True, generator=torch.Generator().manual_seed(seed))
    opt = torch.optim.Adam(student.parameters(), lr=1e-3)
    lossf = nn.CrossEntropyLoss(weight=CLASS_W)
    for _ in range(n_ep):
        student.train()
        for xb, yb in loader:
            opt.zero_grad(); lossf(student(xb.float().to(DEVICE)), yb.to(DEVICE)).backward(); opt.step()
    student.eval()
    return student, raw


for arch, regime in RUN_CELLS:
    for method in METHODS:
        for seed in SEEDS:
            if (arch, regime, method, seed) in done:
                continue
            student, raw = recover_cell(arch, method, regime, seed)
            as_trained = audit(student, arch, "eval")
            batch_stats = audit(student, arch, "batch")
            sd = {k: v.detach().cpu().clone() for k, v in student.state_dict().items()}
            path = CKPT_DIR / f"{arch}_{method}_{regime}_s{seed}.pt"
            torch.save({"state_dict": sd, "architecture": arch, "method": method, "regime": regime, "seed": seed}, path)
            reg_rows.append({"architecture": arch, "method": method, "regime": regime, "seed": seed,
                             "checkpoint": str(path.relative_to(REPO)),
                             "sha256": hashlib.sha256(path.read_bytes()).hexdigest()})
            recalibrate_bn(student)
            recal = audit(student, arch, "eval")
            row = {"architecture": arch, "method": method, "regime": regime, "seed": seed}
            row.update({f"raw_{k}": v for k, v in raw.items() if k in ("awbir", "b2a", "family_f1", "hsr_balanced")})
            row.update({f"trained_{k}": v for k, v in as_trained.items()})
            row.update({f"batch_{k}": v for k, v in batch_stats.items() if k in ("awbir", "b2a", "family_f1", "hsr_balanced")})
            row.update({f"recal_{k}": v for k, v in recal.items()})
            row["trained_is_collapse"] = bool(as_trained["b2a"] > COLLAPSE_B2A and batch_stats["b2a"] <= COLLAPSE_B2A)
            run_rows.append(row)
            pd.DataFrame(run_rows).to_csv(RUNS, index=False)
            pd.DataFrame(reg_rows).to_csv(REG, index=False)
            print(f"{arch} {regime:11s} {method:9s} s{seed}: awbir trained={as_trained['awbir']:.4f} "
                  f"recal={recal['awbir']:.4f} | b2a trained={as_trained['b2a']:.4f} recal={recal['b2a']:.4f}"
                  f"{'  COLLAPSE as-trained' if row['trained_is_collapse'] else ''}")
runs = pd.DataFrame(run_rows); print("rows:", len(runs), "| as-trained collapses:", int(runs["trained_is_collapse"].sum()))


In [ ]:
# Stage 5 - the comparison under both normalisations (N1, N2)
runs = pd.read_csv(OUT / "factorial_runs.csv")
rng = np.random.default_rng(0)


def shares(x):
    grand = x.mean(); ss_tot = ((x - grand) ** 2).sum()
    ss_m = x.shape[0] * ((x.mean(axis=0) - grand) ** 2).sum()
    ss_s = x.shape[1] * ((x.mean(axis=1) - grand) ** 2).sum()
    return float(ss_m / ss_tot), float(ss_s / ss_tot), float((ss_tot - ss_m - ss_s) / ss_tot)


def perm_p(x, B=10000):
    m_obs, s_obs, _ = shares(x); s_null, m_null = [], []
    for _ in range(B):
        xs = x.copy()
        for j in range(xs.shape[1]): xs[:, j] = rng.permutation(xs[:, j])
        s_null.append(shares(xs)[1])
        xm = x.copy()
        for i in range(xm.shape[0]): xm[i, :] = rng.permutation(xm[i, :])
        m_null.append(shares(xm)[0])
    return float(np.mean(np.array(m_null) >= m_obs)), float(np.mean(np.array(s_null) >= s_obs))


def rank_rho(piv):
    ranks = piv.rank(axis=1)
    return float(np.nanmean([ranks.loc[a].corr(ranks.loc[b], method="spearman")
                             for a, b in itertools.combinations(ranks.index, 2)]))


def paired_contrasts(piv):
    out = []
    for a, b in itertools.combinations(METHODS, 2):
        d = (piv[a] - piv[b]).values; m = d.mean(); se = d.std(ddof=1) / np.sqrt(len(d))
        out.append({"contrast": f"{a}-{b}", "mean": float(m), "ci_lo": float(m - 2.776 * se), "ci_hi": float(m + 2.776 * se)})
    return out


analysis, contrast_rows = {}, []
for arch, regime in RUN_CELLS:
    sub = runs[(runs.architecture == arch) & (runs.regime == regime)]
    if len(sub) < len(METHODS) * len(SEEDS):
        print("incomplete, skipped:", arch, regime); continue
    raw_spread = float(sub.groupby("method")["raw_awbir"].first().agg(lambda v: v.max() - v.min()))
    cell = {}
    for norm in ["trained", "recal"]:
        piv = sub.pivot_table(index="seed", columns="method", values=f"{norm}_awbir")[METHODS]
        m_sh, s_sh, i_sh = shares(piv.values); p_m, p_s = perm_p(piv.values)
        rec_spread = float((piv.max(axis=1) - piv.min(axis=1)).mean())
        cell[norm] = {"method_share": m_sh, "seed_share": s_sh, "interaction_residual": i_sh,
                      "p_method": p_m, "p_seed": p_s, "rank_rho": rank_rho(piv),
                      "rei_awbir": float(1 - rec_spread / raw_spread) if raw_spread > 0 else float("nan"),
                      "mean_awbir_by_method": {m: float(v) for m, v in piv.mean().items()}}
        for c in paired_contrasts(piv):
            contrast_rows.append({"architecture": arch, "regime": regime, "normalisation": norm, **c})
        # ground-truth endpoints under the same normalisation
        for metric in ["hsr_balanced", "b2a", "a2b", "family_f1"]:
            pm = sub.pivot_table(index="seed", columns="method", values=f"{norm}_{metric}")[METHODS]
            cell[norm][f"spread_{metric}"] = float((pm.max(axis=1) - pm.min(axis=1)).mean())
            cell[norm][f"mean_{metric}"] = float(pm.values.mean())
    cell["as_trained_collapses"] = int(sub["trained_is_collapse"].sum())
    analysis[f"{arch}/{regime}"] = cell
    print(f"{arch}/{regime}: seed p trained={cell['trained']['p_seed']:.3f} recal={cell['recal']['p_seed']:.3f} | "
          f"method p trained={cell['trained']['p_method']:.3f} recal={cell['recal']['p_method']:.3f} | "
          f"REI trained={cell['trained']['rei_awbir']:.3f} recal={cell['recal']['rei_awbir']:.3f} | "
          f"collapses as-trained={cell['as_trained_collapses']}")
pd.DataFrame(contrast_rows).to_csv(OUT / "paired_contrasts_both_normalisations.csv", index=False)
N1 = sum(1 for c in analysis.values() if c["recal"]["p_seed"] < 0.05) >= 2
N2 = all(not (c["recal"]["p_method"] < 0.05 and c["trained"]["p_method"] >= 0.05) for c in analysis.values())
print("\nN1 (seed effect survives recalibration in >=2 cells):", N1)
print("N2 (no method effect newly detectable after recalibration):", N2)
json.dump(analysis, open(OUT / "both_normalisations_analysis.json", "w"), indent=2)


In [ ]:
# Stage 6 - N3: the dense, unpruned model recovered under the identical conditions
UNP = OUT / "unpruned_control_epochs.csv"
rows = pd.read_csv(UNP).to_dict("records") if UNP.exists() else []
done = {(r["architecture"], r["seed"]) for r in rows}
E_MAX = {"shallow": 8, "deep": 6}
for arch in ["shallow", "deep"]:
    for seed in SEEDS:
        if (arch, seed) in done:
            continue
        torch.manual_seed(seed); np.random.seed(seed)
        model = copy.deepcopy(TEACHERS[arch]).to(DEVICE)
        loader = make_subset_loader(seed)
        opt = torch.optim.Adam(model.parameters(), lr=1e-3); lossf = nn.CrossEntropyLoss(weight=CLASS_W)
        hits = []
        for unit in range(1, E_MAX[arch] + 1):
            model.train()
            for xb, yb in loader:
                opt.zero_grad(); lossf(model(xb.float().to(DEVICE)), yb.to(DEVICE)).backward(); opt.step()
            ev, bs = audit(model, arch, "eval"), audit(model, arch, "batch")
            rows.append({"architecture": arch, "seed": seed, "unit": unit,
                         "eval_b2a": ev["b2a"], "batch_b2a": bs["b2a"], "eval_family_f1": ev["family_f1"],
                         "batch_family_f1": bs["family_f1"], "eval_awbir": ev["awbir"],
                         "bn_var_max": float(max(m.running_var.max().item() for m in model.modules() if isinstance(m, nn.BatchNorm1d)))})
            if ev["b2a"] > COLLAPSE_B2A and bs["b2a"] <= COLLAPSE_B2A:
                hits.append(unit)
        pd.DataFrame(rows).to_csv(UNP, index=False)
        print(f"unpruned {arch} s{seed}: normalisation collapse at {hits or 'none'}")
unp = pd.DataFrame(rows)
N3 = int(((unp.eval_b2a > COLLAPSE_B2A) & (unp.batch_b2a <= COLLAPSE_B2A)).sum()) == 0
print("N3 (unpruned model never collapses):", N3)


In [ ]:
# Stage 7 - N4: recalibration alone on the frozen raw students, no gradient step
RAW = OUT / "raw_recalibration.csv"
rows = []
cells_ = [("shallow", m, b) for m in METHODS for b in (0.25, 0.40, 0.55)] + [("deep", m, 0.40) for m in METHODS]
for arch, method, budget in cells_:
    st = raw_student(arch, method, budget)
    before = audit(st, arch, "eval"); batch = audit(st, arch, "batch")
    recalibrate_bn(st); after = audit(st, arch, "eval")
    rows.append({"architecture": arch, "method": method, "budget": budget,
                 "b2a_before": before["b2a"], "b2a_batchstats": batch["b2a"], "b2a_after": after["b2a"],
                 "a2b_before": before["a2b"], "a2b_after": after["a2b"],
                 "family_f1_before": before["family_f1"], "family_f1_after": after["family_f1"],
                 "hsr_balanced_before": before["hsr_balanced"], "hsr_balanced_after": after["hsr_balanced"]})
    print(f"raw {arch} {method:9s} {budget:.2f}: b2a {before['b2a']:.3f} -> {after['b2a']:.3f} | "
          f"famF1 {before['family_f1']:.3f} -> {after['family_f1']:.3f} | a2b {before['a2b']:.3f} -> {after['a2b']:.3f}")
raw = pd.DataFrame(rows); raw.to_csv(RAW, index=False)
N4 = bool((raw.b2a_after >= COLLAPSE_B2A).all())
print("N4 (recalibration alone rescues no raw student below 10% benign escalation):", N4,
      "| rescued:", int((raw.b2a_after < COLLAPSE_B2A).sum()), "of", len(raw))


In [ ]:
# Stage 8 - the frozen shallow random 40% student: normalisation collapse or genuine degradation?
reg17b = pd.read_csv(R / "17b_calibrated_checkpoint_freeze/shallow_frozen_model_registry.csv")
row = reg17b[(reg17b.method == "random") & (np.isclose(reg17b.target_flops, 0.40))].iloc[0]
ckpt = REPO / str(row["checkpoint"])
result = {"registry_row_found": True, "checkpoint": str(row["checkpoint"]), "checkpoint_exists": ckpt.exists()}
if ckpt.exists():
    file_hash = hashlib.sha256(ckpt.read_bytes()).hexdigest()
    result["file_sha256"] = file_hash
    result["registry_sha256"] = str(row["checkpoint_sha256"])
    result["hash_matches_registry"] = bool(file_hash == str(row["checkpoint_sha256"]))
    if not result["hash_matches_registry"]:
        print("WARNING: file hash differs from the registry entry (hash convention may differ); recorded, continuing")
    payload = torch.load(ckpt, map_location="cpu", weights_only=False)
    if isinstance(payload, dict) and "state_dict" in payload:
        sd = payload["state_dict"]
    elif isinstance(payload, dict) and all(torch.is_tensor(v) for v in payload.values()):
        sd = payload
    else:
        raise RuntimeError(f"unrecognised checkpoint layout: keys {list(payload)[:8] if isinstance(payload, dict) else type(payload)}")
    st = raw_student("shallow", "random", 0.40); st.load_state_dict(sd); st = st.to(DEVICE).eval()
    ev, bs = audit(st, "shallow", "eval"), audit(st, "shallow", "batch")
    st2 = copy.deepcopy(st); recalibrate_bn(st2); rc = audit(st2, "shallow", "eval")
    result.update({"validation_b2a_eval": ev["b2a"], "validation_b2a_batchstats": bs["b2a"],
                   "validation_b2a_recalibrated": rc["b2a"], "validation_family_f1_eval": ev["family_f1"],
                   "validation_family_f1_batchstats": bs["family_f1"], "validation_family_f1_recalibrated": rc["family_f1"],
                   "reading": ("normalisation collapse on validation" if ev["b2a"] > COLLAPSE_B2A and bs["b2a"] <= COLLAPSE_B2A
                               else "not a normalisation collapse on validation; test-time 11.8% unexplained by this check")})
    print(json.dumps(result, indent=2))
else:
    print("frozen checkpoint not found at", ckpt, "- recorded; the test-time 11.8% stays an open item")
json.dump(result, open(OUT / "frozen_random40_dual_mode.json", "w"), indent=2)


In [ ]:
# Stage 9 - NB30 remedy re-run on its saved snapshots with the full outcome set
SNAP = R / "30_remedy_and_trigger/snapshots"
base = pd.read_csv(R / "30_remedy_and_trigger/B_baseline_epochs.csv")
collapsed = base[(base.eval_b2a > COLLAPSE_B2A) & (base.batch_b2a <= COLLAPSE_B2A)]
rows = []
for r in collapsed.itertuples():
    path = SNAP / f"B_{r.architecture}_{r.method}_s{r.seed}_u{r.unit}.pt"
    if not path.exists():
        print("missing snapshot:", path); continue
    p = torch.load(path, map_location="cpu", weights_only=False)
    st = raw_student(p["arch"], p["method"]); st.load_state_dict(p["state_dict"]); st = st.to(DEVICE).eval()
    before = audit(st, r.architecture, "eval")
    # the same fixed calibration slice as every other model in this notebook
    recalibrate_bn(st); after = audit(st, r.architecture, "eval")
    row = {"architecture": r.architecture, "method": r.method, "seed": int(r.seed), "unit": int(r.unit)}
    row.update({f"before_{k}": v for k, v in before.items()}); row.update({f"after_{k}": v for k, v in after.items()})
    rows.append(row)
    print(f"{r.architecture} {r.method} s{r.seed} u{r.unit}: b2a {before['b2a']:.3f}->{after['b2a']:.3f} | "
          f"a2b {before['a2b']:.3f}->{after['a2b']:.3f} | HSR(bal) {before['hsr_balanced']:.3f}->{after['hsr_balanced']:.3f} | "
          f"famF1 {before['family_f1']:.3f}->{after['family_f1']:.3f}")
full = pd.DataFrame(rows); full.to_csv(OUT / "remedy_full_outcomes.csv", index=False)


In [ ]:
# Stage 10 - verdict
analysis = json.load(open(OUT / "both_normalisations_analysis.json"))
unp = pd.read_csv(OUT / "unpruned_control_epochs.csv")
raw = pd.read_csv(OUT / "raw_recalibration.csv")
full = pd.read_csv(OUT / "remedy_full_outcomes.csv") if (OUT / "remedy_full_outcomes.csv").exists() else pd.DataFrame()
frozen = json.load(open(OUT / "frozen_random40_dual_mode.json"))
N1 = sum(1 for c in analysis.values() if c["recal"]["p_seed"] < 0.05) >= 2
N2 = all(not (c["recal"]["p_method"] < 0.05 and c["trained"]["p_method"] >= 0.05) for c in analysis.values())
N3 = int(((unp.eval_b2a > COLLAPSE_B2A) & (unp.batch_b2a <= COLLAPSE_B2A)).sum()) == 0
N4 = bool((raw.b2a_after >= COLLAPSE_B2A).all())
verdict = {
    "arm": "N_consistent_normalisation",
    "N1_seed_effect_survives_recalibration": bool(N1),
    "N2_no_method_effect_newly_detectable": bool(N2),
    "N3_unpruned_never_collapses": bool(N3),
    "N4_recalibration_rescues_no_raw_student": bool(N4),
    "cells": {k: {n: {kk: vv for kk, vv in v[n].items() if kk != "mean_awbir_by_method"} for n in ("trained", "recal")}
              | {"as_trained_collapses": v["as_trained_collapses"]} for k, v in analysis.items()},
    "unpruned_collapse_epochs": int(((unp.eval_b2a > COLLAPSE_B2A) & (unp.batch_b2a <= COLLAPSE_B2A)).sum()),
    "raw_students_rescued_by_recalibration": int((raw.b2a_after < COLLAPSE_B2A).sum()),
    "raw_students_total": int(len(raw)),
    "frozen_random40": frozen,
    "remedy_full_outcomes_median": ({c: float(full[c].median()) for c in full.columns if c.startswith(("before_", "after_"))
                                     and not c.startswith(("before_recall", "after_recall"))} if len(full) else None),
    "prereg": json.load(open(OUT / "N_PREREGISTRATION.json")),
}
(OUT / "N_verdict.json").write_text(json.dumps(verdict, indent=2))
print(json.dumps({k: v for k, v in verdict.items() if k not in ("prereg", "cells")}, indent=2))
for k, v in verdict["cells"].items():
    print(f"{k:22s} seed p {v['trained']['p_seed']:.3f}->{v['recal']['p_seed']:.3f} | method p "
          f"{v['trained']['p_method']:.3f}->{v['recal']['p_method']:.3f} | REI {v['trained']['rei_awbir']:.3f}->{v['recal']['rei_awbir']:.3f}")


In [ ]:
# Stage 11 - figures
runs = pd.read_csv(OUT / "factorial_runs.csv")
fig, axes = plt.subplots(1, len(RUN_CELLS), figsize=(4.6 * len(RUN_CELLS), 3.4), squeeze=False)
for ax, (arch, regime) in zip(axes[0], RUN_CELLS):
    sub = runs[(runs.architecture == arch) & (runs.regime == regime)]
    for method in METHODS:
        d = sub[sub.method == method].sort_values("seed")
        ax.plot(d.seed.astype(str), d.trained_awbir, marker="o", lw=1.1, label=f"{method} as-trained")
        ax.plot(d.seed.astype(str), d.recal_awbir, marker="x", ls="--", lw=1.1, label=f"{method} recalibrated")
    ax.set_title(f"{arch} / {regime}"); ax.set_xlabel("recovery seed"); ax.set_ylabel("AWBIR")
axes[0][0].legend(fontsize=5, ncol=2)
fig.tight_layout(); fig.savefig(OUT / "N_awbir_both_normalisations.png", dpi=200); plt.show()

analysis = json.load(open(OUT / "both_normalisations_analysis.json"))
fig, ax = plt.subplots(figsize=(7.2, 3.4))
labels = list(analysis); xs = np.arange(len(labels))
ax.bar(xs - 0.2, [analysis[k]["trained"]["seed_share"] for k in labels], 0.4, label="seed share, as-trained", color="#dd8452")
ax.bar(xs + 0.2, [analysis[k]["recal"]["seed_share"] for k in labels], 0.4, label="seed share, recalibrated", color="#c44e52")
ax.scatter(xs - 0.2, [analysis[k]["trained"]["method_share"] for k in labels], marker="s", color="k", label="method share, as-trained")
ax.scatter(xs + 0.2, [analysis[k]["recal"]["method_share"] for k in labels], marker="D", color="0.4", label="method share, recalibrated")
ax.set_xticks(xs); ax.set_xticklabels(labels, rotation=20, ha="right", fontsize=8); ax.set_ylabel("share of AWBIR variance")
ax.legend(fontsize=7); ax.set_title("Variance shares before and after uniform recalibration")
fig.tight_layout(); fig.savefig(OUT / "N_variance_shares_both.png", dpi=200); plt.show()

unp = pd.read_csv(OUT / "unpruned_control_epochs.csv")
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for ax, arch in zip(axes, ["shallow", "deep"]):
    for seed in SEEDS:
        d = unp[(unp.architecture == arch) & (unp.seed == seed)].sort_values("unit")
        ax.plot(d.unit, d.eval_b2a, marker="o", lw=1.1, label=f"seed {seed}")
    ax.axhline(COLLAPSE_B2A, color="0.4", ls=":", lw=0.9); ax.set_yscale("symlog", linthresh=1e-3)
    ax.set_title(f"unpruned {arch}: benign escalation per epoch"); ax.set_xlabel("unit"); ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "N_unpruned_control.png", dpi=200); plt.show()
print("figures written ->", OUT)
